# Entraînement ML — Glaucome référable (dossier 0)

Features simples (histogrammes + stats) + Random Forest / régression logistique.  
Métriques : AUC-ROC, sensibilité à 95% spécificité, pAUC.

In [1]:
import sys
from pathlib import Path
# Fonctionne dans Cursor/VS Code (cwd = glaucome) ou Jupyter (cwd = notebooks/)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "train_labels.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from src.data_loading import get_ml_subset_df, get_train_val_test_splits
from src.feature_extraction import build_feature_matrix
from src.evaluation_metrics import roc_auc, sensitivity_at_specificity, partial_auc, print_metrics

## 1. Données et splits

In [2]:
root = PROJECT_ROOT
df = get_ml_subset_df()
train_df, val_df, test_df = get_train_val_test_splits(df)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 12600 Val: 2700 Test: 2700


## 2. Extraction des features (échantillon pour aller plus vite)

In [3]:
# Réduire le nombre d'échantillons pour un run rapide (optionnel)
MAX_TRAIN = 5000
MAX_VAL = 1000
MAX_TEST = 1000

X_train, y_train = build_feature_matrix(train_df, root=root, feature_type="simple", max_samples=MAX_TRAIN)
X_val, y_val = build_feature_matrix(val_df, root=root, feature_type="simple", max_samples=MAX_VAL)
X_test, y_test = build_feature_matrix(test_df, root=root, feature_type="simple", max_samples=MAX_TEST)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

Features (simple): 100%|██████████| 1000/1000 [00:21<00:00, 46.87it/s]

Shapes: (5000, 102) (1000, 102) (1000, 102)


In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

## 3. Entraînement

In [5]:
clf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", random_state=42)
clf.fit(X_train_s, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15, random_state=42)

## 4. Évaluation (validation)

In [6]:
y_val_pred = clf.predict(X_val_s)
y_val_score = clf.predict_proba(X_val_s)[:, 1]
print_metrics(y_val, y_val_pred, y_val_score)

[[971   1]
 [ 28   0]]
AUC-ROC:                    0.6598
pAUC (90-100% spec):        0.5141
Sensibilité @ 95% spec:     0.0714
Sensibilité @ 90% spec:     0.1786


## 5. Évaluation sur le test (une fois satisfait)

In [7]:
y_test_pred = clf.predict(X_test_s)
y_test_score = clf.predict_proba(X_test_s)[:, 1]
print("=== Métriques TEST ===")
print_metrics(y_test, y_test_pred, y_test_score)

=== Métriques TEST ===
[[969   0]
 [ 31   0]]
AUC-ROC:                    0.6798
pAUC (90-100% spec):        0.5501
Sensibilité @ 95% spec:     0.1613
Sensibilité @ 90% spec:     0.2581


## 6. Sauvegarde du modèle (pour éval sur dossier 1, 2, …)

In [ ]:
import joblib

models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, models_dir / "rf_clf.joblib")
joblib.dump(scaler, models_dir / "scaler.joblib")
print("Modèle et scaler sauvegardés dans models/")

Modèle et scaler sauvegardés dans models/
